# Transformer mathematics from first principles

**TL;DR.** This notebook derives a decoder-only transformer one operation at a time using small NumPy
examples. Every mathematical example is paired with a geometric shape: points and arrows for vectors,
rectangles for matrix dimensions, triangles for probability simplices, circles for rotation and
normalization, parallelograms for residual addition, squares for quadratic attention growth, and tiles for
FlashAttention.

The goal is not merely to run a model. The goal is to be able to look at any transformer equation and answer:

1. What are the tensor shapes?
2. What geometric transformation is happening?
3. What information can move between tokens?
4. What consumes parameters, activation memory, KV-cache memory, and compute?

## Learning map

Work top to bottom. Each section introduces exactly one new operation.

1. Tokens become vectors.
2. Vectors are arranged into a matrix.
3. Linear maps create queries, keys, and values.
4. Dot products measure alignment.
5. Scaling, masking, and softmax turn scores into probabilities.
6. Weighted averages move information between tokens.
7. Multiple heads repeat this in parallel subspaces.
8. RoPE rotates vectors according to position.
9. Residuals, normalization, and MLPs form a transformer block.
10. Logits and cross-entropy train next-token prediction.
11. Parameter, activation, attention, and KV-cache memory scale differently.
12. FlashAttention changes memory traffic without changing exact full-attention mathematics.

In [ ]:
from __future__ import annotations

import math

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import Markdown, display

np.set_printoptions(precision=4, suppress=True)

PLOT_TEMPLATE = 'plotly_white'


def show_matrix(matrix, title, *, row_labels=None, column_labels=None, zmid=None):
    matrix = np.asarray(matrix, dtype=float)
    text = np.vectorize(lambda value: f'{value:.3f}')(matrix)
    figure = go.Figure(
        data=go.Heatmap(
            z=matrix,
            text=text,
            texttemplate='%{text}',
            hovertemplate='row=%{y}<br>column=%{x}<br>value=%{z:.6f}<extra></extra>',
            colorscale='Greys',
            zmid=zmid,
        )
    )
    figure.update_layout(
        title=title,
        template=PLOT_TEMPLATE,
        width=max(480, 92 * matrix.shape[1]),
        height=max(360, 72 * matrix.shape[0]),
        xaxis={
            'tickmode': 'array',
            'tickvals': list(range(matrix.shape[1])),
            'ticktext': column_labels or [str(index) for index in range(matrix.shape[1])],
            'side': 'top',
        },
        yaxis={
            'tickmode': 'array',
            'tickvals': list(range(matrix.shape[0])),
            'ticktext': row_labels or [str(index) for index in range(matrix.shape[0])],
            'autorange': 'reversed',
            'scaleanchor': 'x',
        },
    )
    display(figure)


def show_vectors(vectors, labels, title, *, points=None, point_labels=None):
    figure = go.Figure()
    for vector, label in zip(vectors, labels, strict=True):
        vector = np.asarray(vector, dtype=float)
        figure.add_trace(
            go.Scatter(
                x=[0.0, float(vector[0])],
                y=[0.0, float(vector[1])],
                mode='lines+markers+text',
                text=['', label],
                textposition='top center',
                name=label,
            )
        )
    if points is not None:
        points = np.asarray(points, dtype=float)
        figure.add_trace(
            go.Scatter(
                x=points[:, 0],
                y=points[:, 1],
                mode='markers+text',
                text=point_labels,
                textposition='top center',
                name='points',
            )
        )
    figure.update_layout(
        title=title,
        template=PLOT_TEMPLATE,
        xaxis={'zeroline': True, 'scaleanchor': 'y'},
        yaxis={'zeroline': True},
        width=650,
        height=520,
    )
    display(figure)


def show_rectangles(rectangles, title):
    figure = go.Figure()
    cursor = 0.0
    for label, rows, columns in rectangles:
        width = max(1.0, columns / 2.0)
        height = max(1.0, rows / 2.0)
        figure.add_shape(
            type='rect',
            x0=cursor,
            x1=cursor + width,
            y0=-height / 2,
            y1=height / 2,
            line={'width': 2},
            fillcolor='rgba(120,120,120,0.08)',
        )
        figure.add_annotation(
            x=cursor + width / 2,
            y=0,
            text=f'{label}<br>{rows} × {columns}',
            showarrow=False,
        )
        cursor += width + 1.0
    figure.update_layout(
        title=title,
        template=PLOT_TEMPLATE,
        width=900,
        height=420,
        xaxis={'visible': False},
        yaxis={'visible': False, 'scaleanchor': 'x'},
        showlegend=False,
    )
    display(figure)


def show_simplex(probabilities, labels, title):
    probabilities = np.asarray(probabilities, dtype=float)
    vertices = np.array([[0.0, 0.0], [1.0, 0.0], [0.5, math.sqrt(3) / 2]])
    point = probabilities @ vertices
    closed = np.vstack([vertices, vertices[0]])
    figure = go.Figure()
    figure.add_trace(go.Scatter(x=closed[:, 0], y=closed[:, 1], mode='lines', name='probability simplex'))
    figure.add_trace(
        go.Scatter(
            x=vertices[:, 0],
            y=vertices[:, 1],
            mode='markers+text',
            text=labels,
            textposition='top center',
            name='pure-token vertices',
        )
    )
    figure.add_trace(
        go.Scatter(
            x=[point[0]],
            y=[point[1]],
            mode='markers+text',
            text=[f'p={np.round(probabilities, 3)}'],
            textposition='bottom center',
            marker={'size': 15},
            name='distribution',
        )
    )
    figure.update_layout(
        title=title,
        template=PLOT_TEMPLATE,
        width=650,
        height=540,
        xaxis={'visible': False, 'scaleanchor': 'y'},
        yaxis={'visible': False},
    )
    display(figure)


def show_shape_growth(side_lengths, areas, title, x_title):
    figure = go.Figure()
    for index, (side, area) in enumerate(zip(side_lengths, areas, strict=True)):
        offset = index * (max(side_lengths) + 2)
        figure.add_shape(
            type='rect',
            x0=offset,
            x1=offset + side,
            y0=0,
            y1=side,
            line={'width': 2},
            fillcolor='rgba(120,120,120,0.08)',
        )
        figure.add_annotation(
            x=offset + side / 2,
            y=side / 2,
            text=f'{x_title}={side}<br>area={area}',
            showarrow=False,
        )
    figure.update_layout(
        title=title,
        template=PLOT_TEMPLATE,
        width=950,
        height=480,
        xaxis={'visible': False},
        yaxis={'visible': False, 'scaleanchor': 'x'},
    )
    display(figure)


def softmax(values):
    values = np.asarray(values, dtype=float)
    shifted = values - values.max()
    exponentials = np.exp(shifted)
    return exponentials / exponentials.sum()


def layer_norm(vector, epsilon=1e-5):
    vector = np.asarray(vector, dtype=float)
    return (vector - vector.mean()) / np.sqrt(vector.var() + epsilon)


def rms_norm(vector, epsilon=1e-5):
    vector = np.asarray(vector, dtype=float)
    return vector / np.sqrt(np.mean(vector**2) + epsilon)


display(Markdown('**Notebook convention:** every numerical equation is followed by a geometric picture of the same operation.'))

## 1. A token is a point in an embedding space

For vocabulary size \(V\) and model width \(d\), the embedding table is

\[
E \in \mathbb{R}^{V \times d}.
\]

Token \(t_i\) selects one row:

\[
x_i = E[t_i] \in \mathbb{R}^{d}.
\]

Geometrically, each row is a point. Nearby points can encode similar learned usage. The real model may use
thousands of dimensions; the example uses two so the geometry is visible.

In [ ]:
vocabulary = ['cat', 'dog', 'bank', 'river']
embedding_table = np.array([
    [1.2, 1.0],
    [1.0, 1.3],
    [-1.0, 0.9],
    [-0.8, 1.2],
])
token_id = vocabulary.index('cat')
x_cat = embedding_table[token_id]

display(pd.DataFrame(embedding_table, index=vocabulary, columns=['feature 1', 'feature 2']))
show_vectors([], [], 'Embedding rows are points in a vector space', points=embedding_table, point_labels=vocabulary)
assert x_cat.shape == (2,)

## 2. A sequence is a rectangle: \(n\) tokens by \(d\) features

Stack token vectors row by row:

\[
X =
\begin{bmatrix}
x_1^\top \\
x_2^\top \\
\vdots \\
x_n^\top
\end{bmatrix}
\in \mathbb{R}^{n \times d}.
\]

The height of the rectangle is sequence length \(n\). Its width is model dimension \(d\).

In [ ]:
tokens = ['cat', 'sat', 'there']
X = np.array([
    [1.0, 0.2, -0.1, 0.5],
    [0.3, 1.1, 0.4, -0.2],
    [-0.2, 0.1, 0.9, 0.8],
])

show_matrix(X, 'Hidden-state matrix X: one row per token, one column per feature', row_labels=tokens)
show_rectangles([('X', X.shape[0], X.shape[1])], 'Tensor shape as a geometric rectangle')
assert X.shape == (3, 4)

## 3. Linear maps reshape the coordinate system

A linear layer multiplies by a matrix:

\[
Y = XW,
\qquad
X \in \mathbb{R}^{n \times d},
\quad
W \in \mathbb{R}^{d \times d_k},
\quad
Y \in \mathbb{R}^{n \times d_k}.
\]

Geometrically, \(W\) can rotate, stretch, shear, reflect, or project vectors. The inner dimensions \(d\)
must match; the outer dimensions \(n\) and \(d_k\) survive.

In [ ]:
X2 = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
W = np.array([[1.2, 0.6], [-0.4, 0.9]])
Y2 = X2 @ W

display(pd.DataFrame({'input': list(X2), 'output': list(Y2)}))
show_vectors(X2, ['x1', 'x2', 'x3'], 'Before the linear map')
show_vectors(Y2, ['Wx1', 'Wx2', 'Wx3'], 'After rotation, stretching, and shear')
show_rectangles(
    [('X', X2.shape[0], X2.shape[1]), ('W', W.shape[0], W.shape[1]), ('Y', Y2.shape[0], Y2.shape[1])],
    'Matrix multiplication preserves the outer dimensions',
)
assert np.allclose(Y2, X2 @ W)

## 4. Queries, keys, and values are three learned views of the same token

\[
Q=XW_Q,\qquad K=XW_K,\qquad V=XW_V.
\]

A useful mental model:

- a query is the direction in which a token searches;
- a key is the direction a token advertises;
- a value is the payload that will be mixed if the key matches the query.

The token is not split into literal semantic fields. These are three learned coordinate systems.

In [ ]:
W_Q = np.array([[1.0, 0.2], [0.0, 0.8], [0.3, -0.1], [-0.2, 0.4]])
W_K = np.array([[0.7, -0.2], [0.1, 1.0], [0.4, 0.2], [0.0, 0.5]])
W_V = np.array([[0.5, 0.0], [0.2, 0.6], [-0.3, 0.8], [1.0, -0.2]])

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

show_rectangles(
    [('X', 3, 4), ('W_Q', 4, 2), ('Q', 3, 2), ('W_K', 4, 2), ('K', 3, 2), ('W_V', 4, 2), ('V', 3, 2)],
    'Three projections create three geometric views',
)
show_vectors(Q, [f'q({token})' for token in tokens], 'Query space')
show_vectors(K, [f'k({token})' for token in tokens], 'Key space')
show_vectors(V, [f'v({token})' for token in tokens], 'Value space')
assert Q.shape == K.shape == V.shape == (3, 2)

## 5. The dot product measures directional alignment

For query \(q_i\) and key \(k_j\):

\[
q_i^\top k_j = \|q_i\|\,\|k_j\|\cos\theta.
\]

The score is large and positive when arrows point in similar directions, near zero when they are
perpendicular, and negative when they point against each other. It combines angle and magnitude.

In [ ]:
q = np.array([1.5, 0.5])
aligned_key = np.array([1.0, 0.4])
perpendicular_key = np.array([-0.5, 1.5])
opposite_key = np.array([-1.0, -0.2])
keys = [aligned_key, perpendicular_key, opposite_key]
labels = ['aligned', 'perpendicular', 'opposite']
scores = np.array([q @ key for key in keys])

display(pd.DataFrame({'key': labels, 'dot product': scores}))
show_vectors([q, *keys], ['query', *labels], 'Dot products compare angles and lengths')
assert scores[0] > scores[1] > scores[2]

## 6. All query-key comparisons form an \(n \times n\) square

\[
S_{\text{raw}}=QK^\top.
\]

Row \(i\) contains the scores produced by query \(i\) against every key. Column \(j\) contains how every
query scores key \(j\). This square is the source of quadratic context scaling.

In [ ]:
raw_scores = Q @ K.T
show_matrix(
    raw_scores,
    'QKᵀ: every query row compared with every key column',
    row_labels=[f'query: {token}' for token in tokens],
    column_labels=[f'key: {token}' for token in tokens],
    zmid=0,
)
show_rectangles([('Q', 3, 2), ('Kᵀ', 2, 3), ('QKᵀ', 3, 3)], 'Attention scores are an n × n square')
assert raw_scores.shape == (3, 3)

## 7. Divide by \(\sqrt{d_k}\) to control score scale

If query and key coordinates have variance near one, their dot product sums \(d_k\) terms:

\[
\operatorname{Var}(q^\top k) \approx d_k.
\]

Scaled dot-product attention uses

\[
S=\frac{QK^\top}{\sqrt{d_k}}.
\]

Geometrically, dividing by \(\sqrt{d_k}\) contracts the score axis so larger vector spaces do not
automatically push softmax toward nearly one-hot corners.

In [ ]:
rng = np.random.default_rng(7)
dimensions = [4, 16, 64, 256]
rows = []
for dimension in dimensions:
    q_samples = rng.normal(size=(10_000, dimension))
    k_samples = rng.normal(size=(10_000, dimension))
    dots = np.sum(q_samples * k_samples, axis=1)
    rows.append({
        'd_k': dimension,
        'raw standard deviation': dots.std(),
        'scaled standard deviation': (dots / math.sqrt(dimension)).std(),
    })
scale_frame = pd.DataFrame(rows)
display(scale_frame)

figure = go.Figure()
figure.add_trace(go.Bar(x=scale_frame.d_k, y=scale_frame['raw standard deviation'], name='raw'))
figure.add_trace(go.Bar(x=scale_frame.d_k, y=scale_frame['scaled standard deviation'], name='divided by √d_k'))
figure.update_layout(
    title='Scaling contracts the score radius back toward a stable size',
    template=PLOT_TEMPLATE,
    xaxis_title='key/query dimension d_k',
    yaxis_title='standard deviation of dot products',
    barmode='group',
)
display(figure)
assert np.allclose(scale_frame['scaled standard deviation'], 1.0, atol=0.06)

## 8. A causal mask keeps only the lower triangle

A decoder cannot look into the future:

\[
M_{ij}=\begin{cases}
0 & j\le i,\\
-\infty & j>i.
\end{cases}
\]

The visible region is a lower-triangular shape. Token zero sees one key, token one sees two, and so on.

In [ ]:
d_k = Q.shape[1]
scaled_scores = raw_scores / math.sqrt(d_k)
causal_mask = np.triu(np.full((len(tokens), len(tokens)), -np.inf), k=1)
masked_scores = scaled_scores + causal_mask

visible_mask = np.isfinite(masked_scores).astype(float)
show_matrix(visible_mask, 'Causal visibility: the lower triangle is open', row_labels=tokens, column_labels=tokens)
show_matrix(np.where(np.isfinite(masked_scores), masked_scores, np.nan), 'Masked score matrix', row_labels=tokens, column_labels=tokens, zmid=0)
assert np.isneginf(masked_scores[0, 1])

## 9. Softmax maps scores to the probability simplex

For one query row \(s_i\):

\[
a_{ij}=\frac{e^{s_{ij}}}{\sum_m e^{s_{im}}}.
\]

The resulting weights are nonnegative and sum to one. For three keys, every possible distribution lies
inside a triangle. Each corner means all weight on one key; the center means equal weight.

In [ ]:
example_scores = np.array([1.4, 0.5, -0.2])
example_weights = softmax(example_scores)
display(pd.DataFrame({'score': example_scores, 'attention weight': example_weights}, index=tokens))
show_simplex(example_weights, tokens, 'Softmax places the score row inside a probability triangle')
assert np.all(example_weights >= 0)
assert np.isclose(example_weights.sum(), 1.0)

## 10. Attention output is a convex combination of value points

\[
o_i=\sum_{j=1}^{n}a_{ij}v_j.
\]

Because \(a_{ij}\ge0\) and the weights sum to one, \(o_i\) lies inside the convex hull of the value vectors.
In two dimensions with three values, that hull is a triangle.

In [ ]:
value_points = np.array([[0.0, 0.0], [2.0, 0.2], [0.8, 1.8]])
weights = np.array([0.2, 0.5, 0.3])
output_point = weights @ value_points
closed_values = np.vstack([value_points, value_points[0]])

figure = go.Figure()
figure.add_trace(go.Scatter(x=closed_values[:, 0], y=closed_values[:, 1], mode='lines', name='convex hull'))
figure.add_trace(go.Scatter(x=value_points[:, 0], y=value_points[:, 1], mode='markers+text', text=tokens, textposition='top center', name='values'))
figure.add_trace(go.Scatter(x=[output_point[0]], y=[output_point[1]], mode='markers+text', text=['weighted output'], textposition='bottom center', marker={'size': 15}, name='output'))
figure.update_layout(title='The attention result lies inside the value triangle', template=PLOT_TEMPLATE, width=650, height=520, xaxis={'scaleanchor': 'y'})
display(figure)
display(pd.DataFrame({'weight': weights}, index=tokens))
assert np.allclose(output_point, np.sum(weights[:, None] * value_points, axis=0))

## 11. Complete single-head attention

\[
\operatorname{Attention}(X)=
\operatorname{softmax}\!\left(
\frac{(XW_Q)(XW_K)^\top}{\sqrt{d_k}}+M
\right)(XW_V).
\]

Read the equation left to right as a geometry pipeline:

1. project token points into query, key, and value spaces;
2. create an \(n\times n\) square of pairwise alignments;
3. keep the causal triangle;
4. map every row into a probability simplex;
5. use those points as convex-combination coordinates for value vectors.

In [ ]:
attention_weights = np.zeros_like(masked_scores)
for row_index, row in enumerate(masked_scores):
    finite = np.isfinite(row)
    attention_weights[row_index, finite] = softmax(row[finite])
attention_output = attention_weights @ V

show_matrix(attention_weights, 'Causal attention weights: every row sums to one', row_labels=tokens, column_labels=tokens)
show_vectors(V, [f'v({token})' for token in tokens], 'Value points before mixing')
show_vectors(attention_output, [f'o({token})' for token in tokens], 'Output points after attention mixing')
assert np.allclose(attention_weights.sum(axis=1), 1.0)
assert attention_output.shape == V.shape

## 12. Multi-head attention creates parallel subspaces

For head \(r\):

\[
\text{head}_r=
\operatorname{softmax}\!\left(\frac{Q_rK_r^\top}{\sqrt{d_k}}+M\right)V_r.
\]

Then:

\[
\operatorname{MHA}(X)=
\operatorname{Concat}(\text{head}_1,\ldots,\text{head}_h)W_O.
\]

Geometrically, each head gets its own lower-dimensional coordinate plane. Concatenation places those planes
side by side before \(W_O\) mixes them back into model space.

In [ ]:
head_1 = np.array([[1.0, 0.0], [0.7, 0.4], [0.2, 1.0]])
head_2 = np.array([[0.0, 1.0], [-0.5, 0.8], [-1.0, 0.1]])
concatenated = np.concatenate([head_1, head_2], axis=1)
W_O = np.array([
    [0.8, 0.0, 0.2, 0.0],
    [0.0, 0.8, 0.0, 0.2],
    [0.2, 0.0, 0.8, 0.0],
    [0.0, 0.2, 0.0, 0.8],
])
mixed_heads = concatenated @ W_O

show_vectors(head_1, ['h1 token 1', 'h1 token 2', 'h1 token 3'], 'Head 1 coordinate plane')
show_vectors(head_2, ['h2 token 1', 'h2 token 2', 'h2 token 3'], 'Head 2 coordinate plane')
show_rectangles([('head 1', 3, 2), ('head 2', 3, 2), ('concat', 3, 4), ('W_O', 4, 4), ('output', 3, 4)], 'Parallel head rectangles are concatenated and remixed')
assert mixed_heads.shape == (3, 4)

## 13. Position can be represented by rotation: RoPE

In each two-dimensional coordinate pair, rotary position embeddings apply a rotation:

\[
R(\theta_p)=
\begin{bmatrix}
\cos\theta_p & -\sin\theta_p\\
\sin\theta_p & \cos\theta_p
\end{bmatrix},
\qquad
q_p'=R(\theta_p)q_p.
\]

Rotation preserves vector length. Dot products between rotated queries and keys become functions of relative
position because \(R(\theta_i)^\top R(\theta_j)=R(\theta_j-\theta_i)\).

In [ ]:
base_vector = np.array([1.0, 0.25])
positions = np.arange(5)
frequency = math.pi / 8
rotated = []
for position in positions:
    angle = position * frequency
    rotation = np.array([[math.cos(angle), -math.sin(angle)], [math.sin(angle), math.cos(angle)]])
    rotated.append(rotation @ base_vector)
rotated = np.array(rotated)

theta = np.linspace(0, 2 * math.pi, 200)
radius = np.linalg.norm(base_vector)
figure = go.Figure()
figure.add_trace(go.Scatter(x=radius * np.cos(theta), y=radius * np.sin(theta), mode='lines', name='constant-norm circle'))
for position, vector in zip(positions, rotated, strict=True):
    figure.add_trace(go.Scatter(x=[0, vector[0]], y=[0, vector[1]], mode='lines+markers+text', text=['', f'position {position}'], textposition='top center', name=f'p={position}'))
figure.update_layout(title='RoPE moves the same vector around a circle', template=PLOT_TEMPLATE, width=650, height=560, xaxis={'scaleanchor': 'y'})
display(figure)
assert np.allclose(np.linalg.norm(rotated, axis=1), radius)

## 14. Residual connections form a parallelogram

A pre-normalized attention sublayer is commonly written as

\[
U=X+\operatorname{MHA}(\operatorname{Norm}(X)).
\]

For one token vector, residual addition is ordinary vector addition. The old representation and the learned
update form adjacent sides of a parallelogram; the diagonal is the new representation.

In [ ]:
x = np.array([1.1, 0.4])
update = np.array([0.3, 0.9])
residual_result = x + update

figure = go.Figure()
segments = [
    ([0, x[0]], [0, x[1]], 'x'),
    ([0, update[0]], [0, update[1]], 'update'),
    ([x[0], residual_result[0]], [x[1], residual_result[1]], 'translated update'),
    ([update[0], residual_result[0]], [update[1], residual_result[1]], 'translated x'),
    ([0, residual_result[0]], [0, residual_result[1]], 'x + update'),
]
for xs, ys, label in segments:
    figure.add_trace(go.Scatter(x=xs, y=ys, mode='lines+markers', name=label))
figure.update_layout(title='Residual addition is the diagonal of a parallelogram', template=PLOT_TEMPLATE, width=650, height=540, xaxis={'scaleanchor': 'y'})
display(figure)
assert np.allclose(residual_result, [1.4, 1.3])

## 15. LayerNorm centers and rescales; RMSNorm rescales

LayerNorm for one token vector \(x\in\mathbb{R}^d\):

\[
\mu=\frac{1}{d}\sum_jx_j,\qquad
\sigma^2=\frac{1}{d}\sum_j(x_j-\mu)^2,
\]

\[
\operatorname{LN}(x)=\frac{x-\mu}{\sqrt{\sigma^2+\epsilon}}.
\]

RMSNorm omits centering:

\[
\operatorname{RMSNorm}(x)=\frac{x}{\sqrt{\frac{1}{d}\sum_jx_j^2+\epsilon}}.
\]

Geometrically, centering translates the coordinate cloud so its mean is zero. Scaling moves it onto a
standard-radius shell.

In [ ]:
vector = np.array([4.0, 7.0, 1.0, 6.0])
ln_vector = layer_norm(vector)
rms_vector = rms_norm(vector)
norm_frame = pd.DataFrame({'original': vector, 'LayerNorm': ln_vector, 'RMSNorm': rms_vector})
display(norm_frame)

figure = go.Figure()
figure.add_trace(go.Scatter(x=np.arange(len(vector)), y=vector, mode='lines+markers', name='original'))
figure.add_trace(go.Scatter(x=np.arange(len(vector)), y=ln_vector, mode='lines+markers', name='LayerNorm'))
figure.add_trace(go.Scatter(x=np.arange(len(vector)), y=rms_vector, mode='lines+markers', name='RMSNorm'))
figure.add_hline(y=0, line_dash='dash')
figure.update_layout(title='Translation toward zero mean and contraction to a standard scale', template=PLOT_TEMPLATE, xaxis_title='feature coordinate', yaxis_title='value')
display(figure)
assert np.isclose(ln_vector.mean(), 0.0, atol=1e-6)
assert np.isclose(np.mean(ln_vector**2), 1.0, atol=1e-4)
assert np.isclose(np.mean(rms_vector**2), 1.0, atol=1e-4)

## 16. The MLP expands, bends, and compresses each token independently

A conventional feed-forward network is

\[
\operatorname{MLP}(x)=W_2\phi(W_1x+b_1)+b_2.
\]

The first linear map expands \(d\rightarrow d_{ff}\), the nonlinearity bends the space, and the second map
compresses \(d_{ff}\rightarrow d\). Attention mixes across token rows; the MLP transforms each row separately.

In [ ]:
def gelu(values):
    values = np.asarray(values, dtype=float)
    return 0.5 * values * (1 + np.tanh(math.sqrt(2 / math.pi) * (values + 0.044715 * values**3)))

scalar_axis = np.linspace(-3, 3, 300)
figure = go.Figure()
figure.add_trace(go.Scatter(x=scalar_axis, y=scalar_axis, mode='lines', name='linear identity'))
figure.add_trace(go.Scatter(x=scalar_axis, y=gelu(scalar_axis), mode='lines', name='GELU bend'))
figure.update_layout(title='The nonlinearity bends an otherwise flat coordinate line', template=PLOT_TEMPLATE, xaxis_title='pre-activation', yaxis_title='post-activation')
display(figure)

show_rectangles([('token x', 1, 4), ('W₁', 4, 12), ('expanded', 1, 12), ('GELU', 1, 12), ('W₂', 12, 4), ('output', 1, 4)], 'MLP geometry: expand, bend, compress')
assert np.isclose(gelu(np.array([0.0]))[0], 0.0)

## 17. SwiGLU uses one branch as a learned gate

A simplified form is

\[
\operatorname{SwiGLU}(x)=
(xW_a)\odot\operatorname{SiLU}(xW_b),
\qquad
\operatorname{SiLU}(z)=z\sigma(z).
\]

Two projected shapes overlap coordinate by coordinate. The gate branch can shrink, preserve, or reverse parts
of the value branch before the final projection.

In [ ]:
def sigmoid(values):
    return 1 / (1 + np.exp(-values))

value_branch = np.array([1.5, -0.8, 0.5, 2.0])
gate_pre_activation = np.array([2.0, -1.5, 0.0, 0.8])
gate = gate_pre_activation * sigmoid(gate_pre_activation)
gated = value_branch * gate

display(pd.DataFrame({'value branch': value_branch, 'SiLU gate': gate, 'elementwise product': gated}))
figure = go.Figure()
figure.add_trace(go.Bar(x=np.arange(4), y=value_branch, name='value branch'))
figure.add_trace(go.Bar(x=np.arange(4), y=gate, name='gate shape'))
figure.add_trace(go.Bar(x=np.arange(4), y=gated, name='overlap/product'))
figure.update_layout(title='SwiGLU overlaps two coordinate shapes', template=PLOT_TEMPLATE, barmode='group', xaxis_title='hidden coordinate')
display(figure)
assert gated.shape == value_branch.shape

## 18. A complete pre-normalized transformer block

\[
U^{(\ell)}=X^{(\ell)}+
\operatorname{MHA}(\operatorname{Norm}(X^{(\ell)})),
\]

\[
X^{(\ell+1)}=U^{(\ell)}+
\operatorname{MLP}(\operatorname{Norm}(U^{(\ell)})).
\]

The hidden-state rectangle keeps shape \(n\times d\) throughout the block. Internally, attention creates an
\(n\times n\) square and the MLP temporarily creates an \(n\times d_{ff}\) wider rectangle. The code below
executes the complete data path rather than only drawing its shapes.

In [ ]:
def row_rms_norm(matrix, epsilon=1e-5):
    matrix = np.asarray(matrix, dtype=float)
    return matrix / np.sqrt(np.mean(matrix**2, axis=-1, keepdims=True) + epsilon)


def causal_attention_update(matrix, query_weight, key_weight, value_weight, output_weight):
    queries = matrix @ query_weight
    keys = matrix @ key_weight
    values = matrix @ value_weight
    scores = queries @ keys.T / math.sqrt(queries.shape[-1])
    mask = np.triu(np.full((len(matrix), len(matrix)), -np.inf), k=1)
    weights = np.zeros_like(scores)
    for row_index, row in enumerate(scores + mask):
        finite = np.isfinite(row)
        weights[row_index, finite] = softmax(row[finite])
    return (weights @ values) @ output_weight, weights


block_rng = np.random.default_rng(18)
W_attention_out = block_rng.normal(scale=0.3, size=(2, 4))
W_gate_block = block_rng.normal(scale=0.25, size=(4, 12))
W_up_block = block_rng.normal(scale=0.25, size=(4, 12))
W_down_block = block_rng.normal(scale=0.25, size=(12, 4))

attention_input = row_rms_norm(X)
attention_update_block, block_attention_weights = causal_attention_update(
    attention_input, W_Q, W_K, W_V, W_attention_out
)
U = X + attention_update_block
mlp_input = row_rms_norm(U)
gate_pre_activation = mlp_input @ W_gate_block
gate_activation = gate_pre_activation * sigmoid(gate_pre_activation)
mlp_update_block = (gate_activation * (mlp_input @ W_up_block)) @ W_down_block
X_next = U + mlp_update_block

show_rectangles(
    [
        ('X', 3, 4),
        ('Norm(X)', 3, 4),
        ('attention square', 3, 3),
        ('attention update', 3, 4),
        ('residual U', 3, 4),
        ('SwiGLU hidden', 3, 12),
        ('MLP update', 3, 4),
        ('next X', 3, 4),
    ],
    'A transformer block preserves the outer hidden-state rectangle',
)
show_matrix(block_attention_weights, 'Attention weights inside the executed block', row_labels=tokens, column_labels=tokens)
display(pd.DataFrame({
    'token': tokens,
    'input norm': np.linalg.norm(X, axis=1),
    'attention update norm': np.linalg.norm(attention_update_block, axis=1),
    'MLP update norm': np.linalg.norm(mlp_update_block, axis=1),
    'output norm': np.linalg.norm(X_next, axis=1),
}))
assert X_next.shape == X.shape
assert np.allclose(block_attention_weights.sum(axis=1), 1.0)
assert np.all(block_attention_weights[np.triu_indices(len(tokens), k=1)] == 0)
assert np.isfinite(X_next).all()

## 19. Output projection turns one hidden point into vocabulary logits

\[
z_t=h_tW_{\text{vocab}}+b,
\qquad
W_{\text{vocab}}\in\mathbb{R}^{d\times V}.
\]

The hidden vector is projected onto one axis per vocabulary item. Logits are unconstrained coordinates; they
become probabilities only after softmax.

In [ ]:
hidden = np.array([0.8, -0.2, 1.1])
output_matrix = np.array([
    [1.0, 0.2, -0.3],
    [0.1, 0.8, 0.2],
    [0.5, -0.1, 0.9],
])
logits = hidden @ output_matrix
probabilities = softmax(logits)
output_tokens = ['cat', 'sat', 'river']

display(pd.DataFrame({'logit': logits, 'probability': probabilities}, index=output_tokens))
figure = go.Figure()
figure.add_trace(go.Bar(x=output_tokens, y=logits, name='logit coordinates'))
figure.add_trace(go.Bar(x=output_tokens, y=probabilities, name='softmax probabilities'))
figure.update_layout(title='Projection creates free coordinates; softmax folds them into probability mass', template=PLOT_TEMPLATE, barmode='group')
display(figure)
show_simplex(probabilities, output_tokens, 'Vocabulary probabilities live in a simplex')
assert np.isclose(probabilities.sum(), 1.0)

## 20. Cross-entropy measures distance from the correct corner

For the correct next token \(y\):

\[
\mathcal{L}=-\log p(y).
\]

In a three-token simplex, the target is one vertex. Training moves the predicted point toward that vertex.
Probability \(1\) gives loss \(0\); probability approaching \(0\) gives unbounded loss.

In [ ]:
correct_index = 1
correct_probability = probabilities[correct_index]
loss = -math.log(correct_probability)

probability_axis = np.linspace(0.01, 1.0, 300)
figure = go.Figure(go.Scatter(x=probability_axis, y=-np.log(probability_axis), mode='lines'))
figure.add_trace(go.Scatter(x=[correct_probability], y=[loss], mode='markers+text', text=[f'current loss={loss:.3f}'], textposition='top center'))
figure.update_layout(title='Negative log loss curves upward near the wrong edge', template=PLOT_TEMPLATE, xaxis_title='probability assigned to correct token', yaxis_title='loss')
display(figure)
show_simplex(probabilities, output_tokens, 'Prediction point relative to the correct-token vertex')
assert np.isclose(loss, -np.log(correct_probability))

## 21. The softmax-cross-entropy gradient has a simple shape

\[
\frac{\partial\mathcal{L}}{\partial z_v}=p_v-y_v.
\]

The gradient subtracts mass from incorrect coordinates and adds pressure toward the correct coordinate. Its
entries sum to zero, so it is tangent to the probability simplex rather than pointing out of it.

In [ ]:
target = np.zeros_like(probabilities)
target[correct_index] = 1.0
gradient = probabilities - target
display(pd.DataFrame({'probability': probabilities, 'target': target, 'dL/dlogit': gradient}, index=output_tokens))

figure = go.Figure()
figure.add_trace(go.Bar(x=output_tokens, y=gradient, name='gradient'))
figure.add_hline(y=0, line_dash='dash')
figure.update_layout(title='Gradient redistributes mass toward the correct vertex', template=PLOT_TEMPLATE, yaxis_title='p - y')
display(figure)
assert np.isclose(gradient.sum(), 0.0)

## 22. Autoregressive sequence probability is a product of conditional edges

\[
p(x_1,\ldots,x_T)=\prod_{t=1}^{T}p(x_t\mid x_{<t}),
\]

\[
\log p(x_1,\ldots,x_T)=\sum_{t=1}^{T}\log p(x_t\mid x_{<t}).
\]

Geometrically, generation is a path through a branching tree. Every chosen edge contributes a conditional
probability; multiplying edge weights gives the probability of the full path.

In [ ]:
edge_probabilities = [0.7, 0.5, 0.8]
path_probability = math.prod(edge_probabilities)
path_log_probability = sum(math.log(probability) for probability in edge_probabilities)

figure = go.Figure()
nodes = [(0, 0, '<start>'), (1, 0.8, 'the'), (2, 0.2, 'cat'), (3, 0.9, 'sat')]
for left, right, probability in zip(nodes[:-1], nodes[1:], edge_probabilities, strict=True):
    figure.add_trace(go.Scatter(x=[left[0], right[0]], y=[left[1], right[1]], mode='lines+text', text=['', f'p={probability}'], textposition='middle center', showlegend=False))
figure.add_trace(go.Scatter(x=[node[0] for node in nodes], y=[node[1] for node in nodes], mode='markers+text', text=[node[2] for node in nodes], textposition='top center', name='chosen path'))
figure.update_layout(title=f'Path probability = {path_probability:.3f}', template=PLOT_TEMPLATE, xaxis={'visible': False}, yaxis={'visible': False})
display(figure)
assert np.isclose(math.log(path_probability), path_log_probability)

## 23. Model width creates quadratic parameter area

For standard multi-head attention and a conventional MLP with \(d_{ff}=4d\), one block is approximately

\[
N_{\text{attention}}\approx4d^2,
\qquad
N_{\text{MLP}}\approx8d^2,
\qquad
N_{\text{block}}\approx12d^2.
\]

For \(L\) layers:

\[
N_{\text{parameters}}\approx12Ld^2+Vd.
\]

This assumes tied input/output embeddings; untied output weights add another \(Vd\). A SwiGLU block instead has
three MLP matrices, \(3dd_{ff}\), and often chooses a smaller \(d_{ff}/d\) ratio. Doubling width while keeping
architectural ratios fixed doubles both sides of the large matrices, so their area grows fourfold.

In [ ]:
def conventional_parameter_count(layers, width, vocabulary_size):
    attention_per_block = 4 * width**2
    mlp_per_block = 8 * width**2
    embeddings = vocabulary_size * width
    return {
        'attention': layers * attention_per_block,
        'MLP': layers * mlp_per_block,
        'tied embeddings': embeddings,
        'total': layers * (attention_per_block + mlp_per_block) + embeddings,
    }


widths = [1_024, 2_048, 4_096]
parameter_areas = [width**2 for width in [2, 4, 8]]
show_shape_growth([2, 4, 8], parameter_areas, 'Width scaling: matrix area grows as d²', 'relative d')
parameter_rows = []
for width in widths:
    counts = conventional_parameter_count(layers=24, width=width, vocabulary_size=50_000)
    parameter_rows.append({
        'width d': width,
        'attention parameters (B)': counts['attention'] / 1e9,
        'MLP parameters (B)': counts['MLP'] / 1e9,
        'embedding parameters (B)': counts['tied embeddings'] / 1e9,
        'total parameters (B)': counts['total'] / 1e9,
    })
display(pd.DataFrame(parameter_rows))
assert conventional_parameter_count(1, 2_048, 1)['attention'] / conventional_parameter_count(1, 1_024, 1)['attention'] == 4
assert parameter_areas[1] / parameter_areas[0] == 4

## 24. Context length creates quadratic attention-score area

The score matrix is

\[
QK^\top\in\mathbb{R}^{n\times n}.
\]

Therefore standard full-attention score count is \(n^2\). Doubling context length doubles both sides of the
square and quadruples its area. With batch size \(B\) and \(h\) heads, one materialized score tensor contains
\(Bhn^2\) elements. This is about context length, not parameter count.

In [ ]:
relative_contexts = [2, 4, 8]
attention_areas = [length**2 for length in relative_contexts]
show_shape_growth(relative_contexts, attention_areas, 'Context scaling: attention area grows as n²', 'relative n')

actual_contexts = [2_048, 8_192, 32_768]
attention_rows = []
for context in actual_contexts:
    element_count = 32 * context**2  # batch 1, 32 heads
    attention_rows.append({
        'context tokens': context,
        'scores per head': context**2,
        'one BF16 score tensor, 32 heads (GiB)': element_count * 2 / 1024**3,
        'pair multiplier versus 2K': (context / 2_048) ** 2,
    })
display(pd.DataFrame(attention_rows))
assert attention_areas[2] / attention_areas[1] == 4
assert attention_rows[-1]['pair multiplier versus 2K'] == 256

## 25. Training memory is several different shapes added together

A useful decomposition is

\[
M_{\text{train}}\approx
M_{\text{parameters}}+
M_{\text{gradients}}+
M_{\text{optimizer}}+
M_{\text{activations}}.
\]

Parameters, gradients, and optimizer states scale with parameter count. Stored activations scale with batch,
sequence length, width, and layer count; a naïve attention implementation may additionally materialize
\(n\times n\) matrices. A common mixed-precision Adam accounting is roughly 16 bytes per parameter:
2-byte weights, 2-byte gradients, 4-byte master weights, and two 4-byte moment estimates. Exact systems differ
because sharding, quantized optimizers, offload, and precision choices change this budget.

In [ ]:
def gibibytes(byte_count):
    return byte_count / 1024**3


def mixed_precision_adam_components(parameter_count):
    return {
        'BF16 weights': gibibytes(parameter_count * 2),
        'BF16 gradients': gibibytes(parameter_count * 2),
        'FP32 master weights': gibibytes(parameter_count * 4),
        'FP32 Adam moments': gibibytes(parameter_count * 8),
    }


state_rows = []
for parameter_count in [1_000_000_000, 7_000_000_000, 70_000_000_000]:
    components = mixed_precision_adam_components(parameter_count)
    state_rows.append({
        'parameters (B)': parameter_count / 1e9,
        'BF16 inference weights (GiB)': components['BF16 weights'],
        'mixed-precision Adam model state (GiB)': sum(components.values()),
    })
display(pd.DataFrame(state_rows))

example_components = mixed_precision_adam_components(7_000_000_000)
figure = go.Figure()
bottom = 0.0
for name, amount in example_components.items():
    figure.add_trace(go.Bar(x=['7B model'], y=[amount], name=name, base=[bottom]))
    bottom += amount
figure.update_layout(
    title='Mixed-precision Adam model state before activations or sharding',
    template=PLOT_TEMPLATE,
    barmode='stack',
    yaxis_title='GiB',
)
display(figure)
assert np.isclose(sum(example_components.values()), gibibytes(7_000_000_000 * 16))

## 26. Autoregressive inference uses a linearly growing KV-cache rectangle

With cached keys and values, approximate cache memory is

\[
M_{KV}\approx2Lnd_{kv}b,
\]

where \(L\) is layer count, \(n\) is cached sequence length, \(d_{kv}\) is total key/value width per layer,
and \(b\) is bytes per element. Holding other terms fixed, cache size grows linearly in \(n\).

The model still computes attention from the new query to all cached keys at each decoding step, but it does
not recompute old keys and values.

In [ ]:
layers = 32
kv_width = 1024
bytes_per_value = 2
cached_lengths = np.array([1_024, 4_096, 16_384, 65_536])
kv_gib = 2 * layers * cached_lengths * kv_width * bytes_per_value / 1024**3

figure = go.Figure(go.Scatter(x=cached_lengths, y=kv_gib, mode='lines+markers'))
figure.update_layout(title='KV-cache memory forms a line, not an n² curve', template=PLOT_TEMPLATE, xaxis_title='cached tokens n', yaxis_title='GiB')
display(figure)
display(pd.DataFrame({'cached tokens': cached_lengths, 'KV cache GiB': kv_gib}))
assert np.isclose(kv_gib[-1] / kv_gib[-2], cached_lengths[-1] / cached_lengths[-2])

## 27. GQA and MQA narrow the KV-cache rectangle

Standard multi-head attention stores keys and values for every query head. Grouped-query attention shares one
KV head across groups of query heads. Multi-query attention shares a single KV head across all query heads.

If \(h_q\) is the number of query heads and \(h_{kv}\) the number of KV heads, cache width is reduced roughly by

\[
\frac{h_{kv}}{h_q}.
\]

The geometric picture is the same sequence-length rectangle with a narrower feature side.

In [ ]:
query_heads = 32
variants = [('MHA', 32), ('GQA', 8), ('MQA', 1)]
display(pd.DataFrame(
    [{'variant': name, 'KV heads': kv_heads, 'relative cache width': kv_heads / query_heads} for name, kv_heads in variants]
))
show_rectangles(
    [(name, 8, kv_heads) for name, kv_heads in variants],
    'Sharing keys and values narrows the cache rectangle',
)
assert variants[-1][1] / query_heads == 1 / 32

## 28. FlashAttention tiles the \(n\times n\) square

Exact attention can be computed without writing the entire score and probability matrices to high-bandwidth
memory. FlashAttention processes blocks and maintains online softmax statistics.

The mathematical result remains

\[
\operatorname{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}+M\right)V,
\]

and the number of query-key interactions is still quadratic. What changes is the memory-access pattern and
the need to materialize the full square.

In [ ]:
matrix_size = 8
tile_size = 2
figure = go.Figure()
for row in range(0, matrix_size, tile_size):
    for column in range(0, matrix_size, tile_size):
        figure.add_shape(
            type='rect',
            x0=column,
            x1=column + tile_size,
            y0=row,
            y1=row + tile_size,
            line={'width': 1},
            fillcolor='rgba(120,120,120,0.05)',
        )
figure.update_layout(
    title='FlashAttention visits the same square as small tiles',
    template=PLOT_TEMPLATE,
    width=560,
    height=560,
    xaxis={'range': [0, matrix_size], 'scaleanchor': 'y', 'title': 'key blocks'},
    yaxis={'range': [matrix_size, 0], 'title': 'query blocks'},
)
display(figure)
assert (matrix_size // tile_size) ** 2 == 16

## 29. End-to-end worked example with exact numbers

Use two tokens and one two-dimensional head:

\[
Q=
\begin{bmatrix}1&0\\1&1\end{bmatrix},
\quad
K=
\begin{bmatrix}1&0\\0&1\end{bmatrix},
\quad
V=
\begin{bmatrix}10&0\\0&20\end{bmatrix}.
\]

We will calculate every intermediate tensor and display the corresponding shapes.

In [ ]:
Q_toy = np.array([[1.0, 0.0], [1.0, 1.0]])
K_toy = np.array([[1.0, 0.0], [0.0, 1.0]])
V_toy = np.array([[10.0, 0.0], [0.0, 20.0]])
score_toy = Q_toy @ K_toy.T / math.sqrt(2)
mask_toy = np.array([[0.0, -np.inf], [0.0, 0.0]])
masked_toy = score_toy + mask_toy
weights_toy = np.zeros_like(masked_toy)
for row_index, row in enumerate(masked_toy):
    finite = np.isfinite(row)
    weights_toy[row_index, finite] = softmax(row[finite])
output_toy = weights_toy @ V_toy

show_vectors(Q_toy, ['q1', 'q2'], 'Toy query arrows')
show_vectors(K_toy, ['k1', 'k2'], 'Toy key arrows')
show_matrix(score_toy, 'Scaled QKᵀ score square')
show_matrix(np.isfinite(masked_toy).astype(float), 'Causal triangle')
show_matrix(weights_toy, 'Row-wise softmax weights')
show_vectors(V_toy, ['v1', 'v2'], 'Value points')
show_vectors(output_toy, ['o1', 'o2'], 'Final attention outputs')

display(pd.DataFrame(output_toy, index=['token 1', 'token 2'], columns=['output feature 1', 'output feature 2']))
assert np.allclose(weights_toy[0], [1.0, 0.0])
assert np.allclose(weights_toy[1], [0.5, 0.5])
assert np.allclose(output_toy, [[10.0, 0.0], [5.0, 10.0]])

## 30. Final mental model

A decoder-only transformer can be compressed into five geometric operations:

1. **Points:** tokens are vectors in a learned space.
2. **Squares:** attention compares every visible query-key pair.
3. **Triangles/simplexes:** softmax turns scores into probability coordinates.
4. **Convex combinations:** those coordinates mix value points.
5. **Parallelograms and bends:** residuals preserve old vectors while MLPs nonlinearly reshape them.

Keep the scaling laws separate:

\[
\boxed{\text{parameter count}\sim Ld^2}
\]

\[
\boxed{\text{full-attention compute}\sim Ln^2d}
\]

\[
\boxed{\text{KV-cache memory}\sim Lnd_{kv}}
\]

Increasing width, layers, context, batch size, and generated tokens all consume memory, but through different
tensor shapes. The fastest way to reason correctly is to draw the shapes before calculating the bytes.

In [ ]:
scale = np.arange(1, 9)
figure = go.Figure()
figure.add_trace(go.Scatter(x=scale, y=scale**2, mode='lines+markers', name='quadratic area: d² or n²'))
figure.add_trace(go.Scatter(x=scale, y=scale, mode='lines+markers', name='linear edge: n'))
figure.update_layout(
    title='Final geometry: quadratic laws grow as square area; KV cache grows as one edge',
    template=PLOT_TEMPLATE,
    xaxis_title='normalized scale factor',
    yaxis_title='relative growth',
)
display(figure)

show_shape_growth([2, 4, 8], [4, 16, 64], 'Quadratic parameter/attention growth is visible as expanding square area', 'scale')
show_rectangles(
    [('KV n=2', 2, 2), ('KV n=4', 4, 2), ('KV n=8', 8, 2)],
    'Linear KV-cache growth extends one side while feature width stays fixed',
)
assert (scale[-1] ** 2) / (scale[-1]) == scale[-1]